# New fastai text pipeline

We’re rewriting fastai’s text pipeline with two main changes:

- Replace spaCy’s word tokenizer with HF tokenizers (defaulting to GPT-2)
- Add native support for HF Datasets.

This new API should allow users to quickly create `TextDataLoaders` when the dataset follows standard patterns (shown below), while remaining flexible when the dataset doesn't follow those patterns or users want to inspect the pipeline step by step (getting documents, labels, splits, etc.).

We’ll create DataLoaders from datasets organized in 5 different ways:

- **answerdotai/SimpleWiki**: standard HF dataset where each row contains one document.
- Salesforce/wikitext: HF dataset that requires a preprocessing function to transform the raw data into a list of documents.
- **stanfordnlp/imdb**: HF dataset for language model and classification DataLoaders (where we also need a label alongside each document).
- **fastai IMDb**: local folder.
- **Custom data**: Prepare tokenized documents, splits, and optional labels yourself. This is useful when the factory functions don’t fit your data or when you want more control.

In [6]:
from fastai.text.all import *
from fastai.text.tokenizer import *

/Users/hoa/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [7]:
demo_tmp = tempfile.TemporaryDirectory(prefix='fastai-text-dls-')
demo_root = Path(demo_tmp.name)

## SimpleWiki

In [8]:
from datasets import load_dataset


First, let’s load the `articles` subset and look at the first three rows of the training split. For more details, see the [SimpleWiki dataset on Hugging Face](https://huggingface.co/datasets/answerdotai/simplewiki).

In [9]:
wiki_sample = load_dataset('answerdotai/simplewiki', 'articles', split='train[:3]')
wiki_sample.to_pandas()


,article_id,page_id,revision_id,title,timestamp,md,warnings
0,0,1,1806169,April,2026-05-18T12:42:53Z,"**April** (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.\n\nApril always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.\n\n## The Month\n\nApril comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.\n\nApril begins on the same day of the week as Jul...",[]
1,1,2,1806169,August,2026-05-18T12:48:35Z,"**August** (Aug.) is the eighth month of the year in the Gregorian calendar, coming between July and September. It has 31 days. It is named after the Roman emperor Augustus Caesar.\n\nAugust does not begin on the same day of the week as any other month in common years, but begins on the same day of the week as February in leap years. August always ends on the same day of the week as November.\n\n## The Month\n\nThis month was first called *Sextilis* in Latin, because it was the sixth month in the old Roman calendar. The Roman calendar began in March about 735 BC with Romulus. October was t...",[]
2,2,6,693482,Art,2026-08-03T13:45:35Z,"**Art** is a creative activity. It produces a product, an object. **Art** is a diverse range of human activities in creating visual, performing subjects, and expressing the author's thoughts. The product of art is called a **work of art**, for others to experience.\n\nSome art is useful in a practical sense, such as a sculptured clay bowl that can be used. That kind of art is sometimes called a *craft*.\n\nThose who make art are called artists. They hope to affect the emotions of people who experience it. Some people find art relaxing, exciting or informative. Some say people are driven to...",[]


Each row contains one complete document in the `md` column, so we can create `TextDataLoaders` directly from this dataset. To keep downloading and tokenization quick, we’ll use only a small portion of the training split (This also to show how easily the HF Datasets API lets us select a subset of the data)

In [10]:
simple_dls = TextDataLoadersNew.from_dataset('answerdotai/simplewiki', subset='articles',
    text_col='md', train='train[:80]', valid='train[80:100]',
    cache_dir=demo_root/'simplewiki', bs=4, seq_len=32)

 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

Let’s use `show_batch()` to check that the text looks right. We also add two special tokens, `<bos>` and `<eos>`, to mark where each document starts and ends.

In [11]:
simple_dls.show_batch()

<bos>**Art** is a creative activity. It produces a product, an object. **Art** is a diverse range of human activities in creating visual, performing
, the knee, or the thigh. The ones that are tall are usually closed by a zipper or can stretch for putting it on easily. This is because using
ato]] is a "botanical berry"
Kiwi (Actinidia chinensis) 1 Luc Viatour.jpg | A [[K
 changes was that chemistry started to be taught seriously at universities. Before this time, most science was done by individuals working on their own. But now, schools in
**Art** is a creative activity. It produces a product, an object. **Art** is a diverse range of human activities in creating visual, performing subjects
 the knee, or the thigh. The ones that are tall are usually closed by a zipper or can stretch for putting it on easily. This is because using shoe
]] is a "botanical berry"
Kiwi (Actinidia chinensis) 1 Luc Viatour.jpg | A [[Kiw
 was that chemistry started to be taught seriously at universities. Befo

`show_batch()` lets us read the text, but it's not what the model actually see. We can use `one_batch()` to get the input and target tensors. These contain token IDs rather than text

In [14]:
one_batch = simple_dls.one_batch()

In [15]:
one_batch[0]

LMTensorText([[50258,  2025, 12429, 37696,  1174,   318,   281,  4457,  1402,
                3704,   286,  2300,    13,  1439,  3487,  2300,   784,  2279,
                 326,   468,  2347,   784,   318,   925,   286, 23235,    13,
                 770,  3407,  1540,  2340,    11],
              [  976,    11,   772,   996,   262, 15938,  3421,    13,   770,
                2126,   318,  1444,   262, 14903,   286,  2347,    11,  3616,
                2300,  2314,   307,   925,   393,  6572,    11,   691,  3421,
                 656,  1223,   649,    13,   770],
              [ 1913,  3466,   287, 18836,    11, 12217,    11,  2010,  1894,
                  11, 22948,  4652,   290, 22948,  6441,    11,   290, 17706,
                 880,   287, 16259,    11,   686,  5469,   290, 14899,    13,
               10714,  2968,  5701,  2291,  6638],
              [ 2957,   284,   262,  7040,   286,   867,   661,   422, 31261,
                  13,   383, 23897,  9303,  4073,  1049,  1919, 469

We can use `tok.decode` to turn these IDs back to text

In [24]:
simple_dls.tok.decode(one_batch[0][0].tolist(), skip_special_tokens=False)

'<bos>An **atom** is an extremely small piece of matter. All normal matter – everything that has mass – is made of atoms. This includes solids,'

In [22]:
#TODO: Can we avoid tolist when running tok.decode?

## WikiText-103

In this section, we’ll work with the [WikiText-103 dataset](https://huggingface.co/datasets/Salesforce/wikitext). Its format isn’t quite what we need: each document is spread across several rows. We need to merge the rows between article headings (`= Title =`) to form a complete document.

In [26]:
wikitext_sample = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train[:8]')
wikitext_sample.to_pandas()

,text
0,
1,= Valkyria Chronicles III = \n
2,
3,"Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the "" Nameless "" , a penal military unit serving the nation of Gallia during the Second E..."
4,"The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Ozawa . A large team of writers handled the script . The game 's opening theme was sung by May 'n . \n"
5,"It met with positive sales in Japan , and was praised by both Japanese and western critics . After release , it received downloadable content , along with an expanded edition in November of that year . It was also adapted into manga and an original video animation series . Due to low sales of Valkyria Chronicles II , Valkyria Chronicles III was not localized , but a fan translation compatible with the game 's expanded edition was released in 2014 . Media.Vision would return to the franchise with the development of Valkyria : Azure Revolution for the PlayStation 4 . \n"
6,
7,= = Gameplay = = \n


The `get_wiki103_data` function below loops through all rows in the HF dataset and groups rows between article headings `(= Title =)` to form complete documents.



In [25]:
_wiki_heading = re.compile(r'^ = [^=].* = \n$')

def get_wiki103_data(ds):
    "Assemble WikiText-103 articles and their metadata from source lines"
    docs, cur = [], []
    for text in ds['text']:
        if _wiki_heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    if cur: docs.append(''.join(cur))
    texts = [d for d in docs if d.strip()]
    return texts, pd.DataFrame(dict(split=[str(ds.split)]*len(texts)))

In [26]:
wiki_dls = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:3000]', valid='validation',
    cache_dir=demo_root/'wikitext', bs=4, seq_len=32)


 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

In [27]:
wiki_dls.show_batch()

<bos> = SMS Zrínyi = 
 SMS Zrínyi ( " His Majesty 's ship Zrínyi " ) was
 – 2005 television season . As part of the contract , Fey was to develop a prime @-@ time project to be produced by Broadway Video and NBC Universal .
 in July 1916 , Markgraf went into the Baltic for trials . The ship was then temporarily assigned to the I Scouting Group for the fleet operation on 18 –
 elephant 's head , carved tusk , and trunk are seen hanging from the left hand . The second left hand depicts a snake coiled round it . The
 = SMS Zrínyi = 
 SMS Zrínyi ( " His Majesty 's ship Zrínyi " ) was a
 2005 television season . As part of the contract , Fey was to develop a prime @-@ time project to be produced by Broadway Video and NBC Universal . The
 July 1916 , Markgraf went into the Baltic for trials . The ship was then temporarily assigned to the I Scouting Group for the fleet operation on 18 – 19
 's head , carved tusk , and trunk are seen hanging from the left hand . The second left hand depicts a 

## IMDb: classification

When we specify a `label_col` when creating DataLoaders, we get classification DataLoaders that return pairs of texts and labels.


In [30]:
imdb_sample = load_dataset('stanfordnlp/imdb', split='train[:8]')
imdb_sample.to_pandas()

,text,label
0,"I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered ""controversial"" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain politica...",0
1,"""I Am Curious: Yellow"" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don't exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies li...",0
2,"If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />",0
3,"This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been replaced with stupidity. Without going too far on this subject, I would say that follows from the dif...",0
4,"Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that old Peggy Lee song..<br /><br />""Is that all there is??"" ...I was just an early teen when this smoked fish hit the U.S. I was too young to get in the theater (although I did manage to sneak into ""Goodbye Columbus""). Then a screening at a local film museum beckoned - Finally I could see this film, except now I was as old as my parents were when they schlepped to see it!!<br /><br />The ONLY reason this film was not condemned to the anonymous sands of time was because of the obscenity case spa...",0
5,"I would put this at the top of my list of films in the category of unwatchable trash! There are films that are bad, but the worst kind are the ones that are unwatchable but you are suppose to like them because they are supposed to be good for you! The sex sequences, so shocking in its day, couldn't even arouse a rabbit. The so called controversial politics is strictly high school sophomore amateur night Marxism. The film is self-consciously arty in the worst sense of the term. The photography is in a harsh grainy black and white. Some scenes are out of focus or taken from the wrong angle. ...",0
6,"Whoever wrote the screenplay for this movie obviously never consulted any books about Lucille Ball, especially her autobiography. I've never seen so many mistakes in a biopic, ranging from her early years in Celoron and Jamestown to her later years with Desi. I could write a whole list of factual errors, but it would go on for pages. In all, I believe that Lucille Ball is one of those inimitable people who si

In [32]:
imdb_clas = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label',
    train='train[:50]+train[-50:]', valid='test[:20]+test[-20:]',
    cache_dir=demo_root/'imdb-clas', bs=4, seq_len=32)



In [33]:
imdb_clas.show_batch()

,text,category
0,"<bos>I have this film out of the library right now and I haven't finished watching it. It is so bad I am in disbelief. Audrey Hepburn had totally lost her talent by then, although she'd pretty much finished with it in 'Robin and Marian.' This is the worst thing about this appallingly stupid film. It's really only of interest because it was her last feature film and because of the Dorothy Stratten appearance just prior to her homicide.<br /><br />There is nothing but idiocy between Gazzara and his cronies. Little signals and little bows and nods to real screwball comedy of which this is the faintest, palest shadow.<br /><br />Who could believe that there are even some of the same Manhattan environs that Hepburn inhabited so magically and even mythically in 'Breakfast at Tiffany's' twenty years earlier? The soundtrack of old Sinatra songs and the Gershwin song from which the",0
1,"<bos>WARNING: This review contains SPOILERS. Do not read if you don't want some points revealed to you before you watch the film.<br /><br />With a cast like this, you wonder whether or not the actors and actresses knew exactly what they were getting into. Did they see the script and say, `Hey, Close Encounters of the Third Kind was such a hit that this one can't fail.' Unfortunately, it does. Did they even think to check on the director's credentials? I mean, would YOU do a movie with the director of a movie called `Satan's Cheerleaders?' Greydon Clark, who would later go on to direct the infamous `Final Justice,' made this. It makes you wonder how the people of Mystery Science Theater 3000 could hammer `Final Justice' and completely miss out on `The Return.'<br /><br />The film is set in a small town in New Mexico. A little boy and",0
2,"<bos>'The Adventures Of Barry McKenzie' started life as a satirical comic strip in 'Private Eye', written by Barry Humphries and based on an idea by Peter Cook. McKenzie ( 'Bazza' to his friends ) is a lanky, loud, hat-wearing Australian whose two main interests in life are sex ( despite never having had any ) and Fosters lager. In 1972, he found his way to the big screen for the first of two outings. It must have been tempting for Humphries to cast himself as 'Bazza', but he wisely left the job to Barry Crocker ( later to sing the theme to the television soap opera 'Neighbours'! ). Humphries instead played multiple roles in true Peter Sellers fashion, most notably Bazza's overbearing Aunt 'Edna Everage' ( this was before she became a Dame ).<br /><br />You know this is not going to be 'The Importance Of Being Ernest' when its",1
3,"<bos>Who are these ""They""- the actors? the filmmakers? Certainly couldn't be the audience- this is among the most air-puffed productions in existence. It's the kind of movie that looks like it was a lot of fun to shoot TOO much fun, nobody is getting any actual work done, and that almost always makes for a movie that's no fun to watch.<br /><br />Ritter dons glasses so as to hammer home his character's status as a sort of doppleganger of the bespectacled Bogdanovich; the scenes with the breezy Ms. Stratten are sweet, but have an embarrassing, look-guys-I'm-dating-the-prom-queen feel to them. Ben Gazzara sports his usual cat's-got-canary grin in a futile attempt to elevate the meager plot, which requires him to pursue Audrey Hepburn with all the interest of a narcoleptic at an insomnia clinic. In the meantime, the budding couple's respective children (nepotism alert: Bogdanovich's daughters) spew cute and pick up",0


## Fastai IMDb data

When your dataset is saved in local folders, for example, following a `split/label/file` layout, we can use `TextDataLoadersNew.from_folder` to create DataLoaders.


In [34]:
imdb_path = untar_data(URLs.IMDB)

In [35]:
imdb_path.ls()

[Path('/Users/hoa/.fastai/data/imdb/test'), Path('/Users/hoa/.fastai/data/imdb/tmp_clas'), Path('/Users/hoa/.fastai/data/imdb/imdb.vocab'), Path('/Users/hoa/.fastai/data/imdb/unsup'), Path('/Users/hoa/.fastai/data/imdb/README'), Path('/Users/hoa/.fastai/data/imdb/tmp_lm'), Path('/Users/hoa/.fastai/data/imdb/train')]

In [38]:
Path('/Users/hoa/.fastai/data/imdb/test').ls()[:10]

[Path('/Users/hoa/.fastai/data/imdb/test/neg'), Path('/Users/hoa/.fastai/data/imdb/test/pos'), Path('/Users/hoa/.fastai/data/imdb/test/labeledBow.feat')]

In [40]:
folder_clas = TextDataLoadersNew.from_folder(imdb_path, valid='test',
    cache_dir=demo_root/'imdb-files-cache', bs=4, seq_len=32)

 |███████████████████████████████| 100.00% [98/98 00:02<00:00]

In [41]:
folder_clas.show_batch()

,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>When Sam Peckinpah's superlative THE WILD BUNCH (1969) opened the door to outrageous displays of graphic cinematic ultra-violence, it did so with a talented (if whisky-marinated) hand guiding the camera and had a compelling story with characters who had actual depth, but in no time flat there were scores of imitators that fell far from the benchmark set by Peckinpah's epic, and SOLDIER BLUE definitely falls into that category.<br /><br />SOLDIER BLEW, er, BLUE tells the story of foul-mouthed New Yorker Cresta Lee (Candice Bergen) a blonde proto-hippie chick who's been ""rescued"" from two years of ""captivity"" among the Cheyenne and is now being sent to a fort where she'll be reunited with the fiancée she only wants to marry for his money. Also on board the wagon she's traveling in is a shipment of government gold, cash the Cheyenne need to buy guns with, so in short order the",neg
2,"<bos>Okay... it seems like so far, only the Barman fans have commented on this film - time for a counterpoint. Beware, this writeup is *LONG*.<br /><br />For those not in the knowing (mostly the non-Belgians) : Tom Barman, director of this film, is the frontman of dEUS, one of the better known rock bands of the late 90's here in Belgium. Basically, they made a couple of very adventurous and innovative albums and quickly rose to fame on the national scale. Then, egos started hurting and the band basically fell apart, with Barman and a couple of others remaining to go on making albums under the dEUS-monicker. The way it always happens in such cases, the post-breakdown dEUS was a lot tamer and less interesting than the original. They tried to go for an international breakthrough with their album ""The Ideal Crash"" in 1999, presenting a much diluted form of",neg
3,"<bos>""It's all up to you, Ed?"" ""Now you get to play the game."" The day my father left my mother, he took me to see this film with his new girlfriend, on opening day at the Cinerama Dome in Hollywood. I was but fourteen years old. I wasn't even entirely sure what I was seeing, but I will never forgot it, especially the scene with Ned Beatty.<br /><br />Watching it today, thirty-six years later, it still gives me quivers.<br /><br />My father was a huge Burt Reynolds fan, hence the reason for standing in line for almost an hour in to watch what was supposed to be an 'action movie.' I had seen two films at the Dome prior to this: ""Grand Prix"" and ""The Song of Norway."" Well this wasn't exactly Mario Andretti or The Brady Bunch. After leaving the theater, I was shell-shocked.<br /><br />But even at that",pos


## Non standard data

Sometimes, your dataset doesn't follow the standard layout that the factory functions support, or you want to build each piece of the data pipeline (items, labels, splits, etc.) step by step for fine-grained control. In this case, you can prepare the documents, labels, and splits yourself, then use `TextDataLoadersNew.from_docs` to create your DataLoaders.


In this section, we’ll create our own DataLoaders using `TextDataLoadersNew.from_docs` based on a pandas DataFrame. But it can work with any kind of dataset. For example, documents from a folder and labels from a CSV. As long as you can turn your data into a list of documents and labels (or just documents for a language model), you can create DataLoaders.


In [28]:
records = pd.DataFrame([{'body': 'The soup needs more garlic.', 'topic': 'cooking'},
    {'body': 'The team scored twice.', 'topic': 'sport'},
    {'body': 'Bake the bread until golden.', 'topic': 'cooking'},
    {'body': 'The goalkeeper saved the final shot.', 'topic': 'sport'}])
records

,body,topic
0,The soup needs more garlic.,cooking
1,The team scored twice.,sport
2,Bake the bread until golden.,cooking
3,The goalkeeper saved the final shot.,sport


In [29]:
texts = records['body'].tolist()
labels = records['topic'].tolist()
doc_tok = hf_tokenizer()
flat, offsets = tokenize_texts(texts, doc_tok)
docs = TokenDocs(flat, offsets)
doc_splits = ([0, 1], [2, 3])

 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

In [32]:
docs[0]

array([  464, 17141,  2476,   517, 17644,    13], dtype=uint16)

In [35]:
labels

['cooking', 'sport', 'cooking', 'sport']

In [53]:
memory_clas = TextDataLoadersNew.from_docs(docs, doc_tok, doc_splits, labels=labels, bs=2, seq_len=8)
test_eq(memory_clas.vocab[1], ['cooking', 'sport'])
memory_clas.show_batch(max_n=2, trunc_at=30)

,text,category
0,<bos>The soup needs more garlic.<eos>,cooking
1,<bos>The team scored twice.<eos>,sport


## A small ULMFiT training run

In this section, we’ll try a small ULMFiT training run with three stages:

- Stage 1: Train a language model (LM) from scratch on WikiText.
- Stage 2: Fine-tune that LM on the target IMDb texts.
- Stage 3: Fine-tune a classifier using the encoder from stage 2.

This section focuses only on showing how the new DataLoaders API fits into training an NLP model, keeping the training minimal. It doesn’t follow the exact ULMFiT recipe.

### Stage 1: LM from WikiText

Here we create language model DataLoaders from the WikiText-103 dataset, as we showed above. We then pass them to `language_model_learner` and train the model.


In [55]:
wiki_train = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:30]', valid='validation[:20]',
    cache_dir=demo_root/'train-wiki', bs=4, seq_len=32)


In [56]:
lm_config = dict(awd_lstm_lm_config, pad_token=wiki_train.pad_id)
learn = language_model_learner(wiki_train, AWD_LSTM, config=lm_config, pretrained=False,
    metrics=[accuracy, Perplexity()], path=demo_root/'ulmfit')
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         9.498712    8.693745    0.039605  5965.479492 00:01     


### Stage 2: adapt Wiki LM to IMDb

In this section, we create new language model DataLoaders from the IMDb dataset, then replace the DataLoaders from stage 1 with this new one

In [58]:
imdb_train_lm = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text',
    train='train[:8]+unsupervised[:8]', valid='test[:4]',
    cache_dir=demo_root/'train-imdb-lm', bs=4, seq_len=32)
learn.dls = imdb_train_lm

learn.fit(1)
learn.save_encoder('imdb-forward')

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         7.215844    7.963374    0.035397  2873.751709 00:01     


### Stage 3: Classification

In this final stage, we create new classification DataLoaders from the IMDb dataset, create a new learner, load the encoder from stage 2, and fine-tune the classifier.


In [59]:
imdb_train_clas = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label',
    train='train[:8]+train[-8:]', valid='test[:4]+test[-4:]',
    cache_dir=demo_root/'train-imdb-clas', bs=4, seq_len=32)


 |███████████████████████████████| 100.00% [1/1 00:00<00:00]

In [60]:
clas_config = dict(awd_lstm_clas_config, pad_token=imdb_train_clas.pad_id)
learn = text_classifier_learner(imdb_train_clas, AWD_LSTM, config=clas_config, seq_len=32,
    pretrained=False, metrics=accuracy, path=demo_root/'ulmfit')
learn.load_encoder('imdb-forward')
learn.unfreeze()
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  time    
0         0.924771    0.692410    0.500000  00:01     


In [61]:
learn.show_results()

,text,category,category_
0,"<bos>STAR RATING: ***** Saturday Night **** Friday Night *** Friday Morning ** Sunday Night * Monday Morning <br /><br />Former New Orleans homicide cop Jack Robideaux (Jean Claude Van Damme) is re-assigned to Columbus, a small but violent town in Mexico to help the police there with their efforts to stop a major heroin smuggling operation into their town. The culprits turn out to be ex-military, lead by former commander Benjamin Meyers (Stephen Lord, otherwise known as Jase from East Enders) who is using a special method he learned in Afghanistan to fight off his opponents. But Jack has a more personal reason for taking him down, that draws the two men into an explosive final showdown where only one will walk away alive.<br /><br />After Until Death, Van Damme appeared to be on a high, showing he could make the best straight to video films in the action market.",0,1
1,"<bos>Five minutes in, i started to feel how naff this was looking, you've got a completely unheroic hero and his overweight fool of a friend. Seen it all before, yeah right. I was getting ready to be bored out of my mind for a good few hours. This is something i have become quite used to... haven't we all. Then after a few minutes of testosterone fuelled insults and such, the truck appeared. Okay the filming techniques used to make it look fast were clumsy, but who cares! That truck is amazing! Soon however that is taken away again and we're back to the geek and his overweight friend. But now i'm satisfied that at least it won't be too terrible. I then proceed to be amazed again and again by the cleverness of the film. There are so many jokes at their expense, it's like everyone in the world",1,1
2,"<bos>I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn't match the background, and painfully one-dimensional characters cannot be overcome with a 'sci-fi' setting. (I'm sure there are those of you out there who think Babylon 5 is good sci-fi TV. It's not. It's clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It's really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are",0,1
3,"<bos>Worth the entertainment value of a rental, especially if you like action movies. This one features the usual car chases, fights with the great Van Damme kick style, shooting battles with the 40 shell load shotgun, and even terrorist style bombs. All of this is entertaining and competently handled but there is nothing that really blows you away if you've seen your share before.<br /><br />The plot is made interesting by the inclusion of a rabbit, which is clever but hardly profound. Many of the characters are heavily stereotyped -- the angry veterans, the terrified illegal aliens, the crooked cops, the indifferent feds, the bitchy tough lady station head, the crooked politician, the fat federale who looks like he was typecast as the Mexican in a Hollywood movie from the 1940s. All passably acted but again nothing special.<br /><br />I thought the main villains were pretty well done and fairly",0,1


### Backward

Ensembling forward and backward models, where the backward model is trained on text in reverse order, can improve results according to the ULMFiT paper.



We can do this by setting `backwards=True` on each DataLoader. Here’s an example of stage 1 using backward DataLoaders.


In [62]:
wiki_backward = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:30]', valid='validation[:20]',
    cache_dir=demo_root/'train-wiki', backwards=True, bs=4, seq_len=32)

In [63]:
wiki_backward.show_batch()

 written by Seiko Fujibayashi , who had worked on May 'n on previous singles . 
 = = = Release = = = 
<eos>
 handled by Media.Vision. The original scenario was written Kazuki Yamanobe , while the script was written by Hiroyuki Fujii , Koichi Maj
 incursion into Imperial territory , other orders cause certain members of the 422nd great distress . One such member , Gusurg , becomes so enraged that he aband
alkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player
 were written by Seiko Fujibayashi , who had worked on May 'n on previous singles . 
 = = = Release = = = 

 also handled by Media.Vision. The original scenario was written Kazuki Yamanobe , while the script was written by Hiroyuki Fujii , Koichi
 successful incursion into Imperial territory , other orders cause certain members of the 422nd great distress . One such member , Gusurg , becomes so enraged that he ab
 Valkyira C

In [64]:
wiki_backward.tfms

Pipeline: HfTokenizer -> Reverse

In [36]:
    
learn = language_model_learner(wiki_backward, AWD_LSTM, config=lm_config, pretrained=False,
    metrics=[accuracy, Perplexity()], path=demo_root/'ulmfit')
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         9.474694    8.710701    0.042101  6067.494141 00:00     
